In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2204_112806', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2404_230020', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2504_041052', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_2504_092107', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_2504_143159', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_2504_194621', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_2404_120137', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2404_171225']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01022500': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.01 0.85 0.73 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 0.9299 0.7355 ... 0.4174}},
 'camels_01031500': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 0.38 0.32 0.27 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 0.9242 0.64 ... 0.8045 0.6413}},
 'camels_01047000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
   

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}

for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['streamflow_obs']
    sim = xr_ds['streamflow_sim']
    
    # Skip basin if all obs or sim are NaN
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = calculate_metrics(
        obs=obs,
        sim=sim,
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

Skipping camels_06291500 — all observed/simulated values are NaN


,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01022500,0.749536,1.558084,1.248233,0.867014,0.998797,0.875566,0.953097,-0.043508,3.069363,-6.344704,41.908733,0.272727,0.281250,36.178741
camels_01031500,0.800767,2.044064,1.429708,0.847784,0.901590,0.895543,1.050733,0.034540,-8.391373,-16.757488,64.915009,0.272727,0.181818,30.384523
camels_01047000,0.761448,2.855816,1.689916,0.752403,0.802424,0.877956,0.914132,-0.060452,-17.735838,-13.208194,18.773703,0.363636,0.270270,35.831432
camels_01052500,0.841340,1.954236,1.397940,0.867342,0.897413,0.917528,1.016505,0.010619,-9.976896,-10.814296,46.969646,0.166667,0.315789,33.657455
camels_01054200,0.603612,13.159534,3.627607,0.557595,0.617607,0.799534,0.903505,-0.050458,-39.208607,-6.626186,7.239662,0.500000,0.368421,51.915413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_14309500,0.811385,4.402045,2.098105,0.892911,0.979378,0.904165,1.043110,0.021732,1.652160,-13.720020,76.827507,0.454545,0.360000,47.932102
camels_14316700,0.884033,2.615345,1.617203,0.920555,0.947631,0.940259,1.000085,0.000056,-1.292006,-16.529079,15.800031,0.300000,0.241379,29.870966
camels_14325000,0.851026,8.467632,2.909920,0.803636,0.897914,0.926913,0.849018,-0.085442,-6.038246,-12.542130,61.304237,0.222222,0.280000,38.239361


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(f"./ensemble_metrics/{save_name}.csv")

In [7]:
df_metrics.median()

NSE              0.578610
MSE              1.900161
RMSE             1.378463
KGE              0.624649
Alpha-NSE        0.775702
Pearson-r        0.784941
Beta-KGE         0.946225
Beta-NSE        -0.030948
FHV            -22.600198
FMS            -14.938948
FLV             20.895127
Peak-Timing      0.444444
Missed-Peaks     0.500000
Peak-MAPE       51.586184
dtype: float64